# Bedrock AgentCore Gateway를 사용하여 OpenAPI API를 MCP 도구로 변환

## 개요
Bedrock AgentCore Gateway를 사용하면 인프라나 호스팅을 관리할 필요 없이 기존 API를 완전관리형 MCP 서버로 전환할 수 있습니다. JSON 또는 YAML 형식의 OpenAPI 사양을 사용할 수 있습니다. 여기서는 OAuth2로 보호되는 엔터프라이즈 지원 API를 사용하는 고객 서비스 에이전트를 시연합니다.

에이전트를 외부 도구에 연결하는 Gateway 워크플로는 다음 단계로 구성됩니다.
* **Gateway용 도구 생성** - REST API의 OpenAPI 사양과 같은 스키마를 사용하여 도구를 정의합니다. 그러면 Amazon Bedrock AgentCore가 OpenAPI 사양을 파싱하여 Gateway를 생성합니다.
* **Gateway 엔드포인트 생성** - 인바운드 인증이 적용된 MCP 진입점 역할을 하는 Gateway를 생성합니다.
* **Gateway에 대상 추가** - Gateway가 요청을 특정 도구로 라우팅하는 방식을 정의하는 OpenAPI 대상을 구성합니다. OpenAPI 파일에 포함된 모든 API는 MCP 호환 도구가 되며 Gateway 엔드포인트 URL을 통해 사용할 수 있습니다. 각 OpenAPI Gateway 대상에 대해 OAuth를 사용한 아웃바운드 권한 부여를 구성합니다. 
* **에이전트 코드 업데이트** - 에이전트를 Gateway 엔드포인트에 연결하여 통합 MCP 인터페이스를 통해 구성된 모든 도구에 액세스합니다.

![작동 방식](images/openapis-oauth-gateway.png)

### 튜토리얼 세부 정보


| 정보                  | 세부 정보                                                  |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형         | 대화형                                                     |
| AgentCore 구성 요소  | AgentCore Gateway, AgentCore Identity                     |
| 에이전트 프레임워크   | Strands Agent                                             |
| Gateway 대상 유형    | OpenAPI                                                   |
| 에이전트              | 고객 지원 에이전트                                         |
| 인바운드 인증 IdP     | Okta                                                      |
| 아웃바운드 권한 부여  | OAuth                                                     |
| LLM 모델              | Anthropic Claude Haiku 4.5, Amazon Nova Pro              |
| 튜토리얼 구성 요소    | AgentCore Gateway 생성 및 호출                            |
| 튜토리얼 적용 분야    | 산업 전반                                                  |
| 예제 난이도           | 쉬움                                                       |
| 사용된 SDK            | boto3                                                     |

튜토리얼의 첫 번째 부분에서는 몇 가지 AmazonCore Gateway 대상을 생성합니다.

### 튜토리얼 아키텍처
이 튜토리얼에서는 OpenAPI YAML/JSON 파일에 정의된 작업을 MCP 도구로 변환하고 Bedrock AgentCore Gateway에서 호스팅합니다.
시연을 위해 지원 티켓 관련 질문에 답하는 고객 지원 에이전트를 구축합니다. 에이전트는 Zendesk 지원 API의 OpenAPI를 사용합니다. 이 솔루션은 Amazon Bedrock 모델을 사용하는 Langchain Agent를 활용합니다.

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상이 설치된 Jupyter notebook
* uv
* AWS 자격 증명
* Okta
    - client_id
    - client_secret
    - Okta 도메인(예: dev-123456.okta.com)
    - OAuth2 권한 부여 서버 ID(일반적으로 default)
* Okta와 통합된 Zendesk

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# SageMaker notebooks를 사용하지 않는 경우 AWS 자격 증명 설정
import os

# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-east-1")

In [ ]:
import os
import sys

# 현재 스크립트의 디렉터리 가져오기
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우 대체 경로 사용(예: Jupyter)

# utils.py가 있는 디렉터리로 이동(한 수준 위)
utils_dir = os.path.abspath(os.path.join(current_dir, "../.."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

# 이제 utils를 가져올 수 있음
import utils

In [ ]:
#### Gateway가 수임할 IAM 역할 생성

agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

# Gateway 인바운드 권한 부여를 위한 Okta 구성

다음 단계에 따라 Okta에서 OAuth 권한 부여자를 생성합니다. 

* Okta 구독이 없다면 [여기](https://www.okta.com/free-trial/)에서 무료 평가판을 신청합니다.
* Okta Admin console에 로그인합니다. 
* [여기](https://developer.okta.com/docs/guides/implement-grant-type/clientcreds/main/)의 지침에 따라 **Client Credentials** 권한 부여 유형을 사용하는 Application을 생성합니다. 
* Application을 생성한 후 Applications 페이지로 이동하여 방금 생성한 Application을 선택합니다. **Client ID**와 **Client Secret**을 텍스트 편집기에 저장합니다.
* General Settings에서 **Require Demonstrating Proof of Possession (DPoP)**을 비활성화합니다. 
* 왼쪽 탐색 모음을 열고 Security -> API를 선택합니다. 자동으로 생성된 Default Authorization Server를 선택합니다.
* **Audience** 값을 텍스트 편집기에 저장합니다. 
* **Issuer** 값을 텍스트 편집기에 저장합니다(`https://trial-xxxxx.okta.com/oauth2/default`와 유사한 형식이어야 함).
* Custom Scope를 정의합니다. Scopes 탭으로 이동하여 “Add Scope”를 클릭합니다. InvokeGateway라는 범위를 추가합니다. 
* **Access Policies**를 선택합니다. 새 Access Policy를 생성하고 이름을 지정한 다음 Assign to All Clients를 선택합니다. 정책이 생성되면 **Add Rule**을 선택하고 모든 값을 기본값으로 둔 다음 **Create Rule**을 선택합니다. 

# 인바운드 권한 부여용 Okta 권한 부여자를 사용하여 Gateway 생성

In [ ]:
import boto3
from pprint import pprint

gateway_client = boto3.client("bedrock-agentcore-control", region_name=os.environ["AWS_DEFAULT_REGION"])

OKTA_DISCOVERY_URL = "<Your Okta Issuer value>/.well-known/openid-configuration"
OKTA_AUDIENCE = "<The audience value you saved earlier>"

auth_config = {
    "customJWTAuthorizer": {
        "allowedAudience": [OKTA_AUDIENCE],
        "discoveryUrl": OKTA_DISCOVERY_URL,
    }
}
create_response = gateway_client.create_gateway(
    name="OpenAPIOktaGwy2",
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM 역할에는 Gateway 생성/목록 조회/가져오기/삭제 권한이 있어야 함
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway created from sdk with Okta Authorizer",
)
pprint(create_response)
# GatewayTarget 생성에 사용할 GatewayID 가져오기
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]

# Bedrock AgentCore Gateway를 사용하여 Zendesk 지원 API를 MCP 도구로 변환

### 아웃바운드 인증 자격 증명 공급자 생성

In [ ]:
from botocore.config import Config

ZENDESK_DOMAIN = "<Zendek domain url>"
ZENDESK_AUTH_ENDPOINT = "https://<Zendeskl-domain>/oauth/authorizations/new"
ZENDESK_TOKEN_ENDPOINT = "https://<Zendesk-domain>/oauth/tokens"
ZENDESK_CLIENT_ID = ""  # Zendesk OAuth 클라이언트의 client id
ZENDESK_SECRET = ""  # Zendesk OAuth 클라이언트의 client id

sdk_config = Config(
    region_name=os.environ["AWS_DEFAULT_REGION"],
    retries={"max_attempts": 2, "mode": "standard"},
)

acps = boto3.client(
    service_name="bedrock-agentcore-control",
    config=sdk_config,
)

provider_config = {
    "customOauth2ProviderConfig": {
        "oauthDiscovery": {
            "authorizationServerMetadata": {
                "issuer": ZENDESK_DOMAIN,
                "authorizationEndpoint": ZENDESK_AUTH_ENDPOINT,
                "tokenEndpoint": ZENDESK_TOKEN_ENDPOINT,
                "responseTypes": ["token"],
            }
        },
        "clientId": ZENDESK_CLIENT_ID,
        "clientSecret": ZENDESK_SECRET,
    }
}

response = acps.create_oauth2_credential_provider(
    name="ZendeskOAuthTokenCfg",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput=provider_config,
)

pprint(response)
credentialProviderARN = response["credentialProviderArn"]
pprint(f"Egress Credentials provider ARN, {credentialProviderARN}")

### OpenAPI 대상 생성 

#### Zendesk 지원 OpenAPI YAML 파일을 S3에 업로드

In [ ]:
# S3 클라이언트 생성
session = boto3.session.Session()
s3_client = session.client("s3")
sts_client = session.client("sts")

# AWS 계정 ID와 리전 가져오기
account_id = sts_client.get_caller_identity()["Account"]
region = session.region_name
# 파라미터 정의
bucket_name = ""  # OpenAPI JSON 파일을 업로드할 S3 버킷
file_path = "openapi-specs/Zendesk-support-apis.yaml"
object_key = "Zendesk-support-apis.yaml"
# put_object를 사용하여 파일을 업로드하고 응답 읽기
try:
    with open(file_path, "rb") as file_data:
        response = s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=file_data)

    # 계정 ID와 리전을 사용하여 업로드된 객체의 ARN 구성
    openapi_s3_uri = f"s3://{bucket_name}/{object_key}"
    print(f"Uploaded object S3 URI: {openapi_s3_uri}")
except Exception as e:
    print(f"Error uploading file: {e}")

#### Gateway 대상 생성

OpenAPI 파일의 서버 URL이 자체 엔드포인트 URL을 가리키는지 확인합니다. Gateway는 OpenAPI 파일에서 서버 URL을 읽고 해당 엔드포인트를 호출합니다. S3에 업로드하기 전에 반드시 이 값을 변경하세요.

In [ ]:
# OpenAPI 사양 파일의 S3 URI
openapi_s3_target_config = {"mcp": {"openApiSchema": {"s3": {"uri": openapi_s3_uri}}}}

credential_config = [
    {
        "credentialProviderType": "OAUTH",
        "credentialProvider": {
            "oauthCredentialProvider": {
                "providerArn": credentialProviderARN,
                "scopes": ["tickets:read", "read", "tickets:write", "write"],
            }
        },
    }
]

target_name = "DemoOpenAPIGW"
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=target_name,
    description="OpenAPI Target with S3Uri using SDK",
    targetConfiguration=openapi_s3_target_config,
    credentialProviderConfigurations=credential_config,
)

# 결함 보고에 사용할 요청 ID와 타임스탬프 출력. 문제나 결함을 보고할 때 함께 포함할 것
response_metadata = response["ResponseMetadata"]

# Strands Agent에서 Bedrock AgentCore Gateway 호출

Strands Agent는 Model Context Protocol(MCP) 사양을 구현하는 Bedrock AgentCore Gateway를 통해 AWS 도구와 원활하게 통합됩니다. 이 통합을 통해 AI 에이전트와 AWS 서비스 간에 안전하고 표준화된 통신이 가능합니다.

Bedrock AgentCore Gateway는 기본 MCP API인 ListTools와 InvokeTools를 제공하는 프로토콜 호환 Gateway 역할을 합니다. 이러한 API를 통해 모든 MCP 호환 클라이언트 또는 SDK는 사용 가능한 도구를 안전하고 표준화된 방식으로 검색하고 상호 작용할 수 있습니다. Strands Agent가 AWS 서비스에 액세스해야 할 때는 이러한 MCP 표준 엔드포인트를 사용하여 Gateway와 통신합니다.

Gateway 구현은 (MCP 권한 부여 사양)[https://modelcontextprotocol.org/specification/draft/basic/authorization]을 엄격하게 준수하여 강력한 보안과 액세스 제어를 보장합니다. 즉, Strands Agent의 모든 도구 호출은 권한 부여 단계를 거치므로 강력한 기능을 제공하면서도 보안을 유지합니다.

예를 들어 Strands Agent가 MCP 도구에 액세스해야 할 때 먼저 ListTools를 호출하여 사용 가능한 도구를 검색한 다음 InvokeTools를 사용하여 특정 작업을 실행합니다. Gateway는 필요한 모든 보안 검증, 프로토콜 변환 및 서비스 상호 작용을 처리하여 전체 프로세스를 원활하고 안전하게 만듭니다.

이 아키텍처 접근 방식에서는 MCP 사양을 구현하는 모든 클라이언트 또는 SDK가 Gateway를 통해 AWS 서비스와 상호 작용할 수 있으므로, AI 에이전트 통합을 위한 다목적의 미래 지향적 솔루션을 제공합니다.

# 인바운드 권한 부여를 위해 Okta에 액세스 토큰 요청

In [ ]:
print("Requesting the access token from Okta authorizer")
import requests
from requests.auth import HTTPBasicAuth

# 실제 값으로 교체
OKTA_DOMAIN = "Your Okta domain URL"
AUTH_SERVER_ID = "Okta app id"
CLIENT_ID = "<Okta client credentials client id>"
CLIENT_SECRET = "<Okta client credentials secret>"

TOKEN_URL = f"{OKTA_DOMAIN}/oauth2/{AUTH_SERVER_ID}/v1/token"

response = requests.post(
    TOKEN_URL,
    auth=HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET),
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={"grant_type": "client_credentials", "scope": "InvokeGateway"},
)

if response.status_code == 200:
    token = response.json()["access_token"]
    print("Access Token:", token)
else:
    print("Failed to get token:", response.status_code, response.text)

# Bedrock AgentCore Gateway를 통해 Zendesk 지원 API를 사용하는 고객 지원 에이전트에 질문

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})


client = MCPClient(create_streamable_http_transport)

## ~/.aws/credentials에 구성된 IAM 그룹/사용자에게 Bedrock 모델 액세스 권한이 있어야 함
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
import logging


# 루트 strands 로거 구성. 문제를 디버깅하는 경우 DEBUG로 변경
logging.getLogger("strands").setLevel(logging.INFO)

# 로그를 확인할 핸들러 추가
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

with client:
    # listTools 호출
    tools = client.list_tools_sync()
    # 모델과 도구를 사용하여 Agent 생성
    agent = Agent(model=yourmodel, tools=tools)  ## 원하는 모델로 교체 가능
    # print(f"Tools loaded in the agent are {agent.tool_names}")
    # print(f"Tools configuration in the agent are {agent.tool_config}")

    # 샘플 프롬프트로 에이전트 호출. MCP listTools만 호출하여 LLM이 액세스할 수 있는 도구 목록을 가져오며, 아래 코드는 실제로 어떤 도구도 호출하지 않음
    # agent("Hi , can you list all tools available to you")
    agent("Count the number of support tickets")
    # MCP 도구를 명시적으로 호출. MCP 도구 이름과 인수는 AWS Lambda 함수 또는 OpenAPI/Smithy API와 일치해야 함
    result = client.call_tool_sync(
        tool_use_id="count-tickets-1",  # 고유 식별자로 교체 가능
        name="DemoOpenAPIGW___CountTickets",  # AWS Lambda 대상 유형을 기반으로 한 도구 이름이며 대상 이름에 따라 변경됨
    )
    # MCP 도구 응답 출력
    print(f"Tool Call result: {result['content'][0]['text']}")

# 정리
IAM 역할, IAM 정책, 자격 증명 공급자, AWS Lambda 함수, Cognito 사용자 풀, S3 버킷과 같은 추가 리소스도 생성되며 정리 과정에서 수동으로 삭제해야 할 수 있습니다. 이는 실행한 예제에 따라 다릅니다.

## Gateway 삭제(선택 사항)

In [ ]:
import utils

utils.delete_gateway(gateway_client, gatewayID)